# RQ2 — Robustness Evaluation of fine-tuned Qwen3 (Wikidata)

Evaluates the **fine-tuned Qwen3-Embedding-0.6B** (LoRA adapter trained by
`notebooks/finetuning/05_qwen3_contrastive_finetune_square_wikidata.ipynb`)
under the same RQ2 protocol as `04_eval_sparql_RQ2_wikidata.ipynb`:

- **Data:** `datasets/wikidata_testset_RQ2.json` — result-set-preserving perturbation
  pairs (`original`, `perturbed`, `perturbation_type`).
- **Metric:** mean cosine similarity between the embedding of each original query
  and its perturbed version, reported per perturbation type — these are the cells
  of the paper's RQ2 table (`tables/tab-RQ2.tex`), Wikidata block.
- **Sanity check:** MRR before/after perturbation against the RQ1 corpus, with
  the anchor's own embedding excluded from its retrieval pool (same fix as the
  baseline notebook).

The encoder is identical to training: last-token pooling over the PEFT-merged
backbone, L2-normalised — so cosine similarity is a plain dot product.

Outputs a ready-to-paste LaTeX row for the Wikidata block of `tab-RQ2.tex`
(KG-specific operator: BGP splitting) and saves the raw numbers to
`results/robustness/robustness_qwen3_finetuned_wikidata.json`.


In [ ]:
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel
from tqdm.auto import tqdm

# ---- Project path setup (notebook-safe) ----
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

KG              = "wikidata"
MODEL_NAME      = "Qwen/Qwen3-Embedding-0.6B"
ADAPTER_PATH    = os.path.join(_PROJECT_ROOT, "checkpoints", f"qwen3_square_{KG}", "best_adapter")
ROBUSTNESS_FILE = os.path.join(_PROJECT_ROOT, "datasets", f"{KG}_testset_RQ2.json")
TEST_FILE       = os.path.join(_PROJECT_ROOT, "datasets", f"{KG}_testset_RQ1.json")
OUTPUT_DIR      = os.path.join(_PROJECT_ROOT, "results", "robustness")

MAX_LEN    = 512   # same truncation as fine-tuning
BATCH_SIZE = 16

print(f"✓ Project root: {_PROJECT_ROOT}")
print(f"Using device: {DEVICE}")
assert os.path.isdir(ADAPTER_PATH), f"Fine-tuned adapter not found at {ADAPTER_PATH} — run the fine-tuning notebook first."

In [ ]:
# ==========================================
# 1. Fine-tuned Qwen3 encoder — same pooling/normalisation as training
# ==========================================
def last_token_pool(last_hidden_states, attention_mask):
    """Last-token pooling as on the Qwen3-Embedding model card.
    Handles both left- and right-padded batches."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_states.shape[0], device=last_hidden_states.device)
    return last_hidden_states[idx, seq_lens]


class Qwen3FinetunedEncoder:
    """Qwen3-Embedding backbone + fine-tuned LoRA adapter.

    Returns L2-normalised numpy embeddings, matching the training-time
    forward pass of the fine-tuning notebook exactly.
    """

    def __init__(self, model_name, adapter_path, device):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(adapter_path, padding_side="left")
        base = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
        )
        self.model = PeftModel.from_pretrained(base, adapter_path).to(device)
        self.model.eval()

    @torch.no_grad()
    def encode(self, queries, batch_size=BATCH_SIZE, show_progress=True):
        embs = []
        rng = range(0, len(queries), batch_size)
        if show_progress:
            rng = tqdm(rng, desc="Encoding")
        for i in rng:
            batch = list(queries[i:i + batch_size])
            tok = self.tokenizer(batch, padding=True, truncation=True,
                                 max_length=MAX_LEN, return_tensors="pt")
            tok = {k: v.to(self.device) for k, v in tok.items()}
            out = self.model(input_ids=tok["input_ids"],
                             attention_mask=tok["attention_mask"])
            emb = last_token_pool(out.last_hidden_state, tok["attention_mask"])
            embs.append(F.normalize(emb.float(), p=2, dim=1).cpu().numpy())
        return np.vstack(embs)


encoder = Qwen3FinetunedEncoder(MODEL_NAME, ADAPTER_PATH, DEVICE)
print(f"✓ Loaded fine-tuned adapter from {ADAPTER_PATH}")

In [ ]:
# ==========================================
# 2. Metrics — verbatim from 04_eval_sparql_RQ2 (self-exclusion fix included)
# ==========================================
def compute_cosine_sim(emb_orig, emb_pert):
    a = torch.tensor(np.asarray(emb_orig, dtype=np.float32))
    b = torch.tensor(np.asarray(emb_pert, dtype=np.float32))
    return F.cosine_similarity(a, b, dim=1).numpy()


def mrr_against_corpus(anchor_emb, corpus_emb, anchor_idx, pos_idx):
    """
    MRR of a single anchor against the full corpus.
    The anchor's own index is excluded so self-similarity (=1.0)
    does not occupy rank 1 and distort the score.
    """
    scores = sk_cosine(anchor_emb, corpus_emb).flatten()
    scores[anchor_idx] = -np.inf            # exclude the query itself
    best = scores[pos_idx].max()
    rank = 1 + int(np.sum(scores > best))
    return 1.0 / rank

In [ ]:
# ==========================================
# 3. Load data & build corpus — identical to 04_eval_sparql_RQ2_wikidata
# ==========================================
with open(ROBUSTNESS_FILE, 'r', encoding='utf-8') as f:
    pairs = json.load(f)
print(f"Loaded {len(pairs)} Wikidata robustness pairs")
for i, p in enumerate(pairs):
    p['_idx'] = i

test_data_loaded = []
with open(TEST_FILE, 'r', encoding='utf-8') as f:
    try:
        test_data_loaded = json.load(f)
    except json.JSONDecodeError:
        f.seek(0)
        for line in f:
            line = line.strip()
            if line:
                try:
                    test_data_loaded.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
print(f"Loaded {len(test_data_loaded)} items from triplet dataset.")

corpus_queries = []
query_to_idx = {}
anchor_pos_indices = defaultdict(list)

def _add(q):
    q = q.strip()
    if q and q not in query_to_idx:
        query_to_idx[q] = len(corpus_queries)
        corpus_queries.append(q)
    return query_to_idx.get(q)

for item in test_data_loaded:
    anchor = item.get('anchor_query', item.get('query', '')).strip()
    if not anchor:
        continue
    _add(anchor)

    positives = item.get('positive_query', item.get('positive_queries', []))
    if isinstance(positives, str):
        positives = [positives]
    for p in positives:
        p_text = p['query'].strip() if isinstance(p, dict) else p.strip()
        idx = _add(p_text)
        anchor_pos_indices[anchor].append(idx)

    negatives = item.get('negative_query', item.get('negative_queries', []))
    if isinstance(negatives, str):
        negatives = [negatives]
    for n in negatives:
        n_text = n['query'].strip() if isinstance(n, dict) else n.strip()
        _add(n_text)

pert_by_anchor = defaultdict(list)
for p in pairs:
    p['original'] = p['original'].strip()
    p['perturbed'] = p['perturbed'].strip()
    pert_by_anchor[p['original']].append(p)

print("\n[POOL CHECK]")
print(f"  corpus size                = {len(corpus_queries)}")
print(f"  anchors with positives     = {len(anchor_pos_indices)}")
print(f"  anchors with perturbations = {len(pert_by_anchor)}")
_evaluable = [a for a in anchor_pos_indices if a in pert_by_anchor]
print(f"  evaluable anchors (both)   = {len(_evaluable)}")

In [ ]:
# ==========================================
# 4. Evaluate the fine-tuned model — same computation as the baseline notebook
# ==========================================
originals  = [p['original']  for p in pairs]
perturbeds = [p['perturbed'] for p in pairs]

emb_orig   = encoder.encode(originals)
emb_pert   = encoder.encode(perturbeds)
cos_sim    = compute_cosine_sim(emb_orig, emb_pert)
corpus_emb = encoder.encode(corpus_queries)

mrr_orig_total, mrr_pert_total = [], []
for anchor, pos_idx in anchor_pos_indices.items():
    if anchor not in pert_by_anchor or len(pos_idx) == 0:
        continue
    anchor_idx = query_to_idx[anchor]
    a_emb = corpus_emb[anchor_idx:anchor_idx + 1]

    mrr_orig_total.append(
        mrr_against_corpus(a_emb, corpus_emb, anchor_idx, pos_idx)
    )

    versions = []
    for pert_pair in pert_by_anchor[anchor]:
        p_emb = emb_pert[pert_pair['_idx']:pert_pair['_idx'] + 1]
        versions.append(
            mrr_against_corpus(p_emb, corpus_emb, anchor_idx, pos_idx)
        )
    if versions:
        mrr_pert_total.append(float(np.mean(versions)))

mean_orig = float(np.mean(mrr_orig_total)) if mrr_orig_total else 0.0
mean_pert = float(np.mean(mrr_pert_total)) if mrr_pert_total else 0.0

by_type = defaultdict(list)
for p in pairs:
    by_type[p['perturbation_type']].append(p['_idx'])
per_type = {
    pt: {'mean_cos_sim': float(cos_sim[idxs].mean()), 'n_pairs': len(idxs)}
    for pt, idxs in by_type.items()
}

result = {
    'overall_mean_cos_sim': float(cos_sim.mean()),
    'overall_std_cos_sim':  float(cos_sim.std()),
    'mrr_original':  mean_orig,
    'mrr_perturbed': mean_pert,
    'mrr_drop':      mean_orig - mean_pert,
    'n_evaluable_anchors': len(mrr_orig_total),
    'per_type': per_type,
}

print(f"cos_sim={result['overall_mean_cos_sim']:.4f}  "
      f"MRR orig={mean_orig:.4f}  pert={mean_pert:.4f}  "
      f"drop={mean_orig - mean_pert:+.4f}  "
      f"(n={len(mrr_orig_total)} anchors)")

print("\nPer perturbation type (mean cosine similarity | n_pairs):")
for ptype, stats in sorted(per_type.items()):
    print(f"  {ptype:<25}: {stats['mean_cos_sim']:.4f}  (n={stats['n_pairs']})")

In [ ]:
# ==========================================
# 5. LaTeX row for tables/tab-RQ2.tex (Wikidata block) & save JSON
# ==========================================
# Column order of the paper table; KG-specific operator for Wikidata
# is "bgp_split" (BGP splitting).
TABLE_COLUMNS = [
    "variable_renaming",     # Var. Renaming
    "triple_shuffle",        # Triple Reorder
    "redundant_optional",    # Redundant OPTIONAL
    "union_duplication",     # UNION Dup.
    "bgp_split",          # KG-specific†
    "filter_true",           # FILTER(true)
    "distinct_toggle",       # DISTINCT Toggle
]

row_vals = [per_type[c]["mean_cos_sim"] for c in TABLE_COLUMNS]
avg = float(np.mean(row_vals))   # Avg. is the within-KG mean over the 7 columns

cells = " & ".join(f"{v:.3f}" for v in row_vals)
print(f"LaTeX row — Wikidata block of tab-RQ2.tex:")
print(f"Qwen3-Emb-0.6B (fine-tuned) & {cells} & {avg:.3f} \\\\")

os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, f"robustness_qwen3_finetuned_{KG}.json")
with open(out_path, "w") as f:
    json.dump({
        "kg": KG,
        "model": f"{MODEL_NAME} (fine-tuned, best adapter)",
        "adapter": ADAPTER_PATH,
        "protocol": f"notebooks/evaluation-RQ2/Wikidata/04_eval_sparql_RQ2_wikidata.ipynb",
        "table_columns": TABLE_COLUMNS,
        "table_row": {c: per_type[c]["mean_cos_sim"] for c in TABLE_COLUMNS},
        "table_avg": avg,
        **result,
    }, f, indent=2)
print(f"\n✓ Saved to: {out_path}")